# Graph Schema Validation Report for OPFData

This notebook validates whether `power_graph_builder.py` produces a research-usable graph representation.
It checks structure consistency, dimensional consistency, type distributions, information preservation, and normalization behavior.


In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
import json

from power_graph_builder import (
    PowerGraphBuilder,
    NODE_FEAT_DIM, EDGE_FEAT_DIM,
    NODE_RAW_MAX, EDGE_RAW_MAX,
    SOL_NODE_DIM, SOL_EDGE_DIM,
)

DATA_ROOT = Path('power_demo_work/opfdata')
JSON_FILES = sorted(DATA_ROOT.rglob('*.json'))
assert JSON_FILES, f'No OPFData JSON files found in {DATA_ROOT}. Run `python download_dataset.py` from the repository root first.'

print(f'Found {len(JSON_FILES)} OPFData samples under {DATA_ROOT}')
print('First sample:', JSON_FILES[0])


## 1) Raw OPFData structure sanity


In [ ]:
sample_path = JSON_FILES[0]
sample = json.loads(sample_path.read_text(encoding='utf-8'))

print('Top-level keys:', list(sample.keys()))
print('grid.nodes keys:', list(sample['grid']['nodes'].keys()))
print('grid.edges keys:', list(sample['grid']['edges'].keys()))
print('solution.nodes keys:', list(sample.get('solution', {}).get('nodes', {}).keys()))
print('solution.edges keys:', list(sample.get('solution', {}).get('edges', {}).keys()))
print('metadata keys:', list(sample.get('metadata', {}).keys()))


## 2) Transformed graph structure sanity (single sample)


In [ ]:
builder = PowerGraphBuilder(normalize_features=False, include_solution=True, include_links=True)
g = builder.build_graph_from_json(sample_path)

if hasattr(g, 'x'):
    # PyG Data
    x = g.x.tolist()
    edge_index = g.edge_index.tolist()
    edge_attr = g.edge_attr.tolist()
    sol_node = g.sol_node.tolist()
    sol_edge = g.sol_edge.tolist()
    meta = g.meta
    node_type = g.node_type.tolist() if hasattr(g, 'node_type') else None
    edge_type = g.edge_type.tolist() if hasattr(g, 'edge_type') else None
else:
    x = g['x']
    edge_index = g['edge_index']
    edge_attr = g['edge_attr']
    sol_node = g['sol_node']
    sol_edge = g['sol_edge']
    meta = g['meta']
    node_type = g.get('node_type')
    edge_type = g.get('edge_type')

print('x shape:', (len(x), len(x[0]) if x else 0), 'expected last dim', NODE_FEAT_DIM)
print('edge_index shape:', (len(edge_index), len(edge_index[0]) if edge_index else 0))
print('edge_attr shape:', (len(edge_attr), len(edge_attr[0]) if edge_attr else 0), 'expected last dim', EDGE_FEAT_DIM)
print('sol_node shape:', (len(sol_node), len(sol_node[0]) if sol_node else 0), 'expected', SOL_NODE_DIM)
print('sol_edge shape:', (len(sol_edge), len(sol_edge[0]) if sol_edge else 0), 'expected', SOL_EDGE_DIM)
print('meta:', meta)
print('node_type present:', node_type is not None, '| edge_type present:', edge_type is not None)


## 3) Dataset-wide consistency checks


In [ ]:
builder = PowerGraphBuilder(normalize_features=False, include_solution=True, include_links=True)
graphs = builder.batch_process(DATA_ROOT)
print('Processed graphs:', len(graphs))

failures = []
node_type_counter = Counter()
edge_type_counter = Counter()

for idx, g in enumerate(graphs):
    if hasattr(g, 'x'):
        x = g.x.tolist()
        ei = g.edge_index.tolist()
        ea = g.edge_attr.tolist()
        sn = g.sol_node.tolist()
        se = g.sol_edge.tolist()
        meta = g.meta
        ntype = g.node_type.tolist()
        etype = g.edge_type.tolist()
    else:
        x = g['x']
        ei = g['edge_index']
        ea = g['edge_attr']
        sn = g['sol_node']
        se = g['sol_edge']
        meta = g['meta']
        ntype = g['node_type']
        etype = g['edge_type']

    n_nodes = len(x)
    n_edges = len(ea)

    checks = {
        'x_dim': all(len(r) == NODE_FEAT_DIM for r in x),
        'ea_dim': all(len(r) == EDGE_FEAT_DIM for r in ea),
        'sn_dim': all(len(r) == SOL_NODE_DIM for r in sn),
        'se_dim': all(len(r) == SOL_EDGE_DIM for r in se),
        'edge_index_rows': len(ei) == 2,
        'edge_index_len': len(ei[0]) == n_edges and len(ei[1]) == n_edges,
        'meta_nodes': meta.get('n_nodes') == n_nodes,
        'meta_edges': meta.get('n_edges') == n_edges,
        'node_type_len': len(ntype) == n_nodes,
        'edge_type_len': len(etype) == n_edges,
    }

    if not all(checks.values()):
        failures.append((idx, checks, meta.get('source_file', 'unknown')))

    node_type_counter.update(ntype)
    edge_type_counter.update(etype)

print('Consistency failures:', len(failures))
if failures:
    print('First failure:', failures[0])

node_type_names = {0: 'bus', 1: 'generator', 2: 'load', 3: 'shunt'}
edge_type_names = {0: 'ac_line', 1: 'transformer', 2: 'generator_link', 3: 'load_link', 4: 'shunt_link'}

print('Node type distribution:')
for k in sorted(node_type_counter):
    print(f'  {k} ({node_type_names.get(k)}): {node_type_counter[k]}')

print('Edge type distribution:')
for k in sorted(edge_type_counter):
    print(f'  {k} ({edge_type_names.get(k)}): {edge_type_counter[k]}')


## 4) Information-preservation checks


In [ ]:
# Compare raw JSON counts against graph meta for one example
raw_nodes = sample['grid']['nodes']
raw_edges = sample['grid']['edges']

raw_counts = {
    'n_bus': len(raw_nodes.get('bus', [])),
    'n_gen': len(raw_nodes.get('generator', [])),
    'n_load': len(raw_nodes.get('load', [])),
    'n_shunt': len(raw_nodes.get('shunt', [])),
    'n_ac_line': len(raw_edges.get('ac_line', {}).get('senders', [])),
    'n_transformer': len(raw_edges.get('transformer', {}).get('senders', [])),
    'n_generator_link': len(raw_edges.get('generator_link', {}).get('senders', [])),
    'n_load_link': len(raw_edges.get('load_link', {}).get('senders', [])),
    'n_shunt_link': len(raw_edges.get('shunt_link', {}).get('senders', [])),
}

print('Raw counts:')
for k, v in raw_counts.items():
    print(f'  {k}: {v}')

print('\nGraph meta counts:')
for k in ['n_bus','n_gen','n_load','n_shunt','n_nodes','n_edges']:
    print(f'  {k}: {meta.get(k)}')
print('  edge_type_counts:', meta.get('edge_type_counts'))

# Validate one-hot integrity (type block should be one-hot)
x_type_ok = all(abs(sum(row[NODE_RAW_MAX:]) - 1.0) < 1e-6 for row in x)
ea_type_ok = all(abs(sum(row[EDGE_RAW_MAX:]) - 1.0) < 1e-6 for row in edge_attr)
print('\nType one-hot validity:')
print('  node type one-hot valid:', x_type_ok)
print('  edge type one-hot valid:', ea_type_ok)


## 5) Normalization behavior: per-graph vs dataset-level


In [ ]:
# Per-graph mode
g_graph = PowerGraphBuilder(normalize_features=True, normalization_mode='graph').build_graph_from_json(sample_path)

# Dataset-level mode
b_ds = PowerGraphBuilder(normalize_features=False, normalization_mode='dataset')
raw_graphs = b_ds.batch_process(DATA_ROOT)
b_ds.fit_normalizers(raw_graphs)
b_ds.normalize_features = True
g_ds = b_ds.build_graph_from_json(sample_path)

def to_arrays(g):
    if hasattr(g, 'x'):
        return g.x.tolist(), g.edge_attr.tolist()
    return g['x'], g['edge_attr']

xg, eg = to_arrays(g_graph)
xd, ed = to_arrays(g_ds)

print('Per-graph x mean first 3 raw cols:', [round(sum(r[j] for r in xg)/len(xg), 4) for j in range(3)])
print('Dataset-level x mean first 3 raw cols:', [round(sum(r[j] for r in xd)/len(xd), 4) for j in range(3)])
print('Per-graph edge mean first 3 raw cols:', [round(sum(r[j] for r in eg)/len(eg), 4) for j in range(3)])
print('Dataset-level edge mean first 3 raw cols:', [round(sum(r[j] for r in ed)/len(ed), 4) for j in range(3)])
print('Shared normalizers fitted:', b_ds.has_both_fitted_normalizers())


## 6) Research interpretation and known limitations

### Improvements validated in this repo
- Link edges are no longer collapsed into one generic type; `generator_link`, `load_link`, and `shunt_link` are preserved.
- Node/edge integer type vectors (`node_type`, `edge_type`) are exported in addition to one-hot type bits.
- Optional merged dynamic features (`x_dyn`, `edge_attr_dyn`) are available when `merge_solution=True`.
- Normalization explicitly supports `normalization_mode='dataset'` for reproducible train/val/test workflows.
- Raw-only normalization avoids scaling one-hot type bits.

### Remaining limitations
- The representation is still a homogeneous graph with padded shared feature blocks; semantics differ by type for the same column index.
- Link edges still carry no raw electrical parameters (type-only).
- Dynamic solutions are optional and can leak labels if used improperly in predictive tasks; users must control task setup carefully.
- The pipeline does not yet include a dedicated heterogeneous-graph object (e.g., per-type node/edge stores).
